In [2]:
import pandas as pd
import os
from pathlib import Path
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
os.getcwd()

'/home/susip4/komorebi_project/src'

In [4]:
DATA_PATH = Path.cwd().parent/'data'

In [5]:
df_withdrawals = pd.read_parquet(DATA_PATH/"zrive_advertiser_withdrawals.parquet") #bajas solicitadas
df_dim = pd.read_parquet(DATA_PATH/"zrive_dim_advertiser.parquet") #Información del cliente (perfil, provincia, etc.).
df_snapshot = pd.read_parquet(DATA_PATH/"zrive_fct_monthly_snapshot_advertiser.parquet") #actividad mensual

In [6]:
advertisers = df_dim
snapshot    = df_snapshot
withdrawals = df_withdrawals

# ============================================================
# 1. EXPLORACIÓN BÁSICA
# Ver cuántas filas/columnas tiene cada tabla y sus columnas
# ============================================================

In [7]:
for nombre, df in [("advertisers", advertisers),
                   ("snapshot", snapshot),
                   ("withdrawals", withdrawals)]:
    print(f"\n--- {nombre} ---")
    print(f"  Filas: {len(df)}, Columnas: {df.shape[1]}")
    print(f"  Columnas: {list(df.columns)}")
    print(f"  Nulos por columna:\n{df.isnull().sum()}")


--- advertisers ---
  Filas: 7076, Columnas: 8
  Columnas: ['advertiser_zrive_id', 'province_id', 'updated_at', 'advertiser_province', 'advertiser_group_id', 'min_start_contrato_date', 'max_start_contrato_nuevo_date', 'contrato_churn_date']
  Nulos por columna:
advertiser_zrive_id                 0
province_id                         0
updated_at                          0
advertiser_province                 0
advertiser_group_id              5753
min_start_contrato_date             0
max_start_contrato_nuevo_date    1868
contrato_churn_date              3182
dtype: int64

--- snapshot ---
  Filas: 96829, Columnas: 23
  Columnas: ['advertiser_zrive_id', 'period_int', 'monthly_contracted_ads', 'monthly_published_ads', 'monthly_unique_published_ads', 'monthly_distinct_ads', 'monthly_oro_ads', 'monthly_plata_ads', 'monthly_destacados_ads', 'monthly_pepitas_ads', 'monthly_shows', 'monthly_visits', 'monthly_leads', 'monthly_total_phone_views', 'monthly_total_calls', 'monthly_total_emails',

# ============================================================
# 2. RANGO DE FECHAS
# Comprobamos en qué fechas tenemos datos en cada tabla
# ============================================================

In [8]:
# Convertimos las fechas a tipo fecha (datetime)
advertisers["min_start_contrato_date"] = pd.to_datetime(
    advertisers["min_start_contrato_date"], errors="coerce"
)
 
# snapshot usa period_int en formato YYYYMM (ej: 202401 = enero 2024)
snapshot["period_int"] = snapshot["period_int"].astype(str)
snapshot["fecha"] = pd.to_datetime(snapshot["period_int"], format="%Y%m")
 
withdrawals["withdrawal_creation_date"] = pd.to_datetime(
    withdrawals["withdrawal_creation_date"], errors="coerce"
)
 
print("\n--- RANGO DE FECHAS ---")
print(f"advertisers  min_start_contrato_date: "
      f"{advertisers['min_start_contrato_date'].min()} "
      f"→ {advertisers['min_start_contrato_date'].max()}")
print(f"snapshot     period_int:              "
      f"{snapshot['fecha'].min()} "
      f"→ {snapshot['fecha'].max()}")
print(f"withdrawals  withdrawal_creation_date: "
      f"{withdrawals['withdrawal_creation_date'].min()} "
      f"→ {withdrawals['withdrawal_creation_date'].max()}")


--- RANGO DE FECHAS ---
advertisers  min_start_contrato_date: 2009-10-16 00:00:00 → 2025-06-20 00:00:00
snapshot     period_int:              2023-01-01 00:00:00 → 2025-05-01 00:00:00
withdrawals  withdrawal_creation_date: 2012-06-19 07:12:34 → 2025-05-30 11:29:06


# ============================================================
# 3. BAJAS DEFINITIVAS
# Filtramos las bajas "reales" según las reglas del enunciado
# ============================================================

In [9]:
razones_no_definitivas = [
    "Upselling-cambio de contrato",
    "Cambio a Bundle Online",
    "Cambio de Contrato/propuesta/producto"
]
 
bajas_definitivas = withdrawals[
    (withdrawals["withdrawal_type"] == "TOTAL") &
    (withdrawals["withdrawal_status"] != "Denegada") &
    (~withdrawals["withdrawal_reason"].isin(razones_no_definitivas))
].copy()
 
print(f"\n--- BAJAS DEFINITIVAS ---")
print(f"  Total bajas en tabla:      {len(withdrawals)}")
print(f"  Bajas definitivas:         {len(bajas_definitivas)}")
print(f"  Anunciantes con baja def.: {bajas_definitivas['advertiser_zrive_id'].nunique()}")


--- BAJAS DEFINITIVAS ---
  Total bajas en tabla:      22668
  Bajas definitivas:         15941
  Anunciantes con baja def.: 5569


# ============================================================
# 4. CICLOS DE ACTIVIDAD POR USUARIO
# Para cada usuario miramos cuándo está activo (has_active_contract = 1)
# ============================================================

In [11]:
# Ordenamos el snapshot por usuario y mes
snapshot_sorted = snapshot.sort_values(["advertiser_zrive_id", "fecha"])
 
# Para cada usuario, detectamos cambios 1->0 (baja) y 0->1 (renovación)
snapshot_sorted["contrato_anterior"] = snapshot_sorted.groupby(
    "advertiser_zrive_id"
)["has_active_contract"].shift(1)
 
# Cambio de 1 a 0: posible baja
bajas_detectadas = snapshot_sorted[
    (snapshot_sorted["contrato_anterior"] == 1) &
    (snapshot_sorted["has_active_contract"] == 0)
]
 
# Cambio de 0 a 1: renovación o nueva propuesta
renovaciones_detectadas = snapshot_sorted[
    (snapshot_sorted["contrato_anterior"] == 0) &
    (snapshot_sorted["has_active_contract"] == 1)
]

In [12]:
print(f"\n--- CICLOS DETECTADOS EN SNAPSHOT ---")
print(f"  Meses con posible baja (1→0):       {len(bajas_detectadas)}")
print(f"  Meses con renovación/inicio (0→1):  {len(renovaciones_detectadas)}")


--- CICLOS DETECTADOS EN SNAPSHOT ---
  Meses con posible baja (1→0):       905
  Meses con renovación/inicio (0→1):  342


# ============================================================
# 5. USUARIOS A DESCARTAR
# Usuarios cuyo min_start_contrato_date es ANTERIOR al snapshot
# (no tenemos su historial completo)
# ============================================================

In [13]:
fecha_inicio_snapshot = snapshot["fecha"].min()
 
usuarios_antiguos = advertisers[
    advertisers["min_start_contrato_date"] < fecha_inicio_snapshot
]

In [16]:
print(f"\n--- USUARIOS CON INICIO ANTES DEL SNAPSHOT ---")
print(f"  Inicio del snapshot:  {fecha_inicio_snapshot.date()}")
print(f"  Usuarios 'antiguos':  {len(usuarios_antiguos)} "
      f"({100*len(usuarios_antiguos)/len(advertisers):.1f}% del total)")


--- USUARIOS CON INICIO ANTES DEL SNAPSHOT ---
  Inicio del snapshot:  2023-01-01
  Usuarios 'antiguos':  3348 (47.3% del total)
